# geo1263 – How good is our drone map? Accuracy against RTK points and the NRW terrain model

**Unit:** geo1260 Drone mapping · **Slides:** `geo1260_Drone_Imagery_WebODM` (part *Results*) · **Data:** field data of the drone day (RTK points, orthophoto, surface model) · **Tools:** `geo1263_ODM_Tools.py` · **Class / homework**

On the field day the drone flew one mapping mission at about 25 m with RTK corrections from SAPOS. WebODM computed an orthophoto and a digital surface model (DSM) from the photos **without any ground control points**: only the camera positions of the drone were used. On the ground, 31 points were measured with RTK (Emlid, SAPOS): 21 permanent markers of the EOLab team (`P1`–`P21`) and 10 panels laid out by students (`S3` … `S70`).

None of these points was used in the processing, so all 31 are **check points**: an honest test of the map. In this notebook you

1. read and check the RTK points,
2. sort out the coordinate reference systems and the two kinds of height,
3. compare the drone DSM with the RTK heights,
4. compare both with the official terrain model DGM1 of Geobasis NRW,
5. investigate a suspicious point, and
6. decide whether ground control points would have been worth the effort.

## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import geo1263_ODM_Tools as ot             # given tools (same folder as this notebook)

# Field data of the drone day: on the JupyterHub the shared folder, on a laptop the Moodle
# download saved into the folder data/ next to this notebook.
CANDIDATES = [Path.home() / "shared" / "data" / "opengeo" / "geo1260", Path("data")]
FIELD = next((d for root in CANDIDATES if root.exists() for d in sorted(root.glob("gfl_*"))[-1:]), None)
assert FIELD, "field data not found: see the task sheet (shared folder or Moodle download)"
POINTS = FIELD / "gcp" / f"{FIELD.name}_ground_points.gpkg"
ORTHO = next((FIELD / "webodm").glob("*orthophoto*.tif"))
DSM = FIELD / "webodm" / "dsm.tif"
OUT = Path("geo1263_out")                   # your results
OUT.mkdir(exist_ok=True)
print(FIELD, POINTS.exists(), ORTHO.name, DSM.exists(), sep="\n")

## 1. The RTK points

The points were exported from WebODM, where the teaching assistant had loaded them as two overlay layers. `ot.read_points` puts both layers into one table: `id` (`P…` permanent marker, `S…` student panel), `set`, `E`, `N`, `h` (ellipsoidal height), `surface` (what the point lies on), `solution`, `rms_lat`, `rms_h` (accuracy estimated by the receiver).

In [ ]:
pts = ot.read_points(POINTS)
pts.head()

**Exercise 1.1:** check the points before you use them. How many points per set? Are all solutions `FIX`? What are the largest lateral and vertical RMS values? Write `quality(pts)` that returns a small table per `set` with the columns `n`, `fix` (number of FIX solutions), `rms_lat_max`, `rms_h_max`.

In [ ]:
# YOUR CODE HERE

print(quality(pts))
ot.check_points(pts);

*Note:* the RMS values are the receiver's own estimate of its precision during the measurement. They say nothing about a wrong pole height, a point measured in the wrong place, or a panel that moved before the flight. That is why we still need independent checks.

**A first look:** the points on the orthophoto (the cell reads a reduced copy, 20 cm pixels).

In [ ]:
from rasterio.enums import Resampling
with rasterio.open(ORTHO) as src:
    f = 10                                               # 2 cm -> 20 cm
    img = src.read([1, 2, 3], out_shape=(3, src.height // f, src.width // f), resampling=Resampling.average)
    b = src.bounds
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.moveaxis(img, 0, -1), extent=(b.left, b.right, b.bottom, b.top))
for s, c in [("permanent_ground_markers", "deepskyblue"), ("student_gcps", "red")]:
    p = pts[pts["set"] == s]
    ax.scatter(p["E"], p["N"], s=60, facecolors="none", edgecolors=c, linewidths=2, label=s)
    for _, r in p.iterrows():
        ax.annotate(r["id"], (r["E"], r["N"]), xytext=(4, 4), textcoords="offset points", color=c, fontsize=9)
ax.set_xlim(pts.E.min() - 15, pts.E.max() + 15); ax.set_ylim(pts.N.min() - 15, pts.N.max() + 15)
ax.ticklabel_format(useOffset=False, style="plain"); ax.tick_params(axis="x", rotation=30)
ax.legend(loc="lower left"); ax.set_title("RTK points on the drone orthophoto")
fig.savefig(OUT / "geo1263_points.png", dpi=150)

## 2. Which coordinate reference system?

**Exercise 2.1:** print the CRS of the orthophoto and of the DSM (`rasterio.open(...).crs`) and the first corner of each (`.bounds`). Then answer:

1. Which EPSG code does the DSM report? Which one do the RTK points use (`pts.crs`)?
2. Both codes are UTM zone 32N. What is the difference between them?
3. The corners of orthophoto and DSM are almost the same numbers as the RTK coordinates. Should you reproject the DSM to EPSG:25832 before you compare? Why (not)?

In [ ]:
# YOUR CODE HERE


✍️ *Your answer:*

## 3. Two kinds of height

GNSS measures **ellipsoidal heights** $h$ (above the GRS80 ellipsoid). Maps and the NRW elevation models use **DHHN2016** heights $H$ (above the quasigeoid, "above sea level"). The difference is the height of the quasigeoid $N$: $h = H + N$. In Germany $N$ comes from the model GCG2016 of the BKG (a PROJ grid); `ot.to_dhhn(E, N, h)` applies it.

Our RTK points and the drone DSM are both **ellipsoidal** (the drone used SAPOS RTK; WebODM does not change the heights).

**Exercise 3.1:** add the columns `H` (DHHN2016) and `N_qg = h - H` to `pts`. How large is $N$ at the Green Fab Lab, and how much does it change across our area? What would happen if you compared the DSM with an NRW model without this step?

In [ ]:
# YOUR CODE HERE
print(pts["N_qg"].describe().round(4))
ot.check_heights(pts);

✍️ *Your answer:*

## 4. The drone DSM against the RTK heights

`ot.sample_dsm(path, E, N, radius)` returns the median of the DSM in a small square around each point (radius in metres). The DSM has 2 cm pixels, so `radius=0.07` takes about 7 × 7 pixels: enough to smooth single noisy pixels, small enough to stay on the marker.

**Exercise 4.1:** add the column `dsm` (DSM height at each point) and `dz = dsm - h` (error of the map in height, map minus reference). Which height of the point do you have to use, `h` or `H`? Then write `my_stats(df, col, by)`: per group `by`, the columns `n`, `mean`, `std`, `RMSE` with $RMSE = \sqrt{\text{mean}(\Delta^2)}$. Apply it to `dz` per `set` and per `surface`.

In [ ]:
# YOUR CODE HERE

by_set = my_stats(pts, "dz", "set")
by_surface = my_stats(pts, "dz", "surface")
ot.check_stats(by_surface, pts, "dz", "surface")
print((by_set * 100).round(1).assign(n=by_set["n"]), "\n")            # cm
print((by_surface * 100).round(1).assign(n=by_surface["n"]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = ["paving", "gravel", "asphalt", "grass"]
for k, s in enumerate(order):
    d = pts[pts["surface"] == s]
    x = k + np.random.default_rng(1).uniform(-0.15, 0.15, len(d))
    ax.scatter(x, d["dz"] * 100, c=np.where(d["id"].str[0] == "P", "deepskyblue", "red"), s=50)
    for xi, (_, r) in zip(x, d.iterrows()):
        ax.annotate(r["id"], (xi, r["dz"] * 100), xytext=(5, -3), textcoords="offset points", fontsize=8)
ax.axhline(0, color="grey", lw=1)
ax.set_xticks(range(len(order)), order); ax.set_ylabel("DSM − RTK [cm]")
ax.set_title("Height error of the drone DSM (blue: permanent markers, red: student panels)")
fig.tight_layout(); fig.savefig(OUT / "geo1263_dz_surface.png", dpi=150)

**Questions 4.2**

1. How large is the height error of a DSM made **without any ground control points**? Compare with the RTK precision (`rms_h`).
2. The student panels look worse than the permanent markers (`by_set`). Is it the people? Look at `by_surface` and the plot.
3. On grass the DSM is higher than the RTK point. Give two physical reasons.

✍️ *Your answer:*

## 5. Against the official terrain model DGM1

Geobasis NRW publishes the terrain model **DGM1** (bare ground, 1 m grid, from airborne laser scanning, heights DHHN2016). It knows nothing about our flight or our receivers: an independent reference for both.

`ot.nrw_tile("dgm1", e_km, n_km)` downloads a 1 km tile once (lower left corner in km). Our points lie in two tiles. `ot.sample_bilinear(path, E, N)` interpolates a coarse grid bilinearly (the nearest 1 m cell can be 70 cm away).

**Exercise 5.1:** add the columns `dgm1` (DGM1 height at each point; use the right tile for each point: `N < 5708000` → tile `5707`), `rtk_dgm1 = H - dgm1` and `dsm_dgm1 = (dsm - N_qg) - dgm1`. Show both with `my_stats` per `surface`.

In [ ]:
# YOUR CODE HERE
for col in ("rtk_dgm1", "dsm_dgm1"):
    t = my_stats(pts, col, "surface")
    print(col); print((t * 100).round(1).assign(n=t["n"]), "\n")
pts.drop(columns="geometry").to_csv(OUT / "geo1263_points_heights.csv", index=False)

**Questions 5.2**

1. RTK and DGM1 were measured years apart with completely different methods. How well do they agree? Is there a common offset?
2. Why is `dsm_dgm1` on grass larger than `rtk_dgm1` on grass? Use the words *surface model* and *terrain model*.
3. What does this comparison tell you about the georeferencing of our drone result?

✍️ *Your answer:*

## 6. A suspicious point

One point is about 1.8 m higher than all the others. A typical cause in the field is a wrong pole height (entered 0 instead of 1.8 m, or the other way round).

**Exercise 6.1:** find the point (`pts.sort_values(...)`). Before you call it an error, look at its surroundings: sample the DSM (`radius=0.07`) and the DGM1 (`sample_bilinear`) along a west–east and a south–north profile, from −10 m to +10 m around the point every 0.5 m, and plot the four profiles (DSM in DHHN2016: subtract `N_qg`). Is the point wrong?

In [ ]:
# YOUR CODE HERE


✍️ *Your answer:*

## 7. Would GCPs have been worth it?

**Questions 7**

1. Our check found 1–2 cm in height on hard ground and a few cm horizontally (look at the markers on the orthophoto). What could GCPs still improve, and at what cost (laying out, measuring, clicking in the photos)?
2. When would you still use GCPs with this drone? Think of: no SAPOS/no mobile data, older drone without RTK, a volume or deformation study, a customer who needs proof.
3. The ASPRS standard asks for at least 30 check points for an accuracy statement. We have 31. Why so many? What changes if you only had 3?
4. For next year: which points would you use as GCPs and which as check points, and which kind of target would you lay out where?

✍️ *Your answer:*

## Stretch tasks
### 7.1 Horizontal check in QGIS
Open the orthophoto (assign EPSG:25832, see §2) and `geo1263_out/geo1263_points.gpkg` (written below). For 5 points of your choice, read the coordinates of the marker centre on the orthophoto (status bar, or digitise a new point layer) and compute the horizontal error map − RTK ($\Delta E$, $\Delta N$, $\sqrt{\Delta E^2 + \Delta N^2}$).

### 7.2 WebODM with GCPs
In the WebODM project of the course: start a copy of the RGB task **with** GCPs (GCP interface: 5–6 points on hard ground at the edges and in the middle; all others stay check points). Repeat §4 with the new `dsm.tif`. Is it better than 1–2 cm? (Processing takes about 2 hours: start it early.)

### 7.3 Drone DSM against DOM1
Geobasis NRW also publishes the surface model **DOM1**. Resample the drone DSM to the 1 m grid of DOM1 (`rasterio.warp.reproject`, `Resampling.average`), subtract (drone in DHHN2016!) and map the difference. Where is it close to 0, where not, and why?

In [ ]:
# a GeoPackage does not tell h from H (column names are case-insensitive): rename for the export
pts.rename(columns={"h": "h_ell", "H": "H_dhhn2016"}).to_file(OUT / "geo1263_points.gpkg", layer="points", driver="GPKG")

In [ ]:
# YOUR CODE HERE


✍️ *Your answer:*

## Hand in
Upload the notebook (with outputs) and `geo1263_out/geo1263_dz_surface.png` to Moodle.

## Sources
- RTK points, photos, orthophoto and DSM: own field data (EOLab-HSRW, course use only).
- DGM1 and DOM1: Geobasis NRW, Datenlizenz Deutschland – Zero – Version 2.0 (opengeodata.nrw.de).
- Quasigeoid GCG2016: BKG, PROJ grid `de_bkg_gcg2016.tif` (cdn.proj.org, CC BY 4.0).
- OpenDroneMap / WebODM documentation (docs.opendronemap.org, docs.webodm.org).
- ASPRS Positional Accuracy Standards for Digital Geospatial Data, Edition 2 (2023, Version 2 2024).
- EPSG registry: EPSG:25832, EPSG:32632, EPSG:7837 (epsg.org).